# LILY WAN 2.2 — P100 FAST STUDIO
Run Cell 1. If it installs Torch and restarts, run Cell 1 again. When it says PASS, run Cell 2. Cell 2 installs a pinned ComfyUI runtime, downloads the official Wan 2.2 TI2V 5B files, and launches a phone-friendly Gradio studio. TURBO deliberately uses a small diffusion workload for speed.


In [ ]:
import os, subprocess, sys, time
def sh(a,check=True):
 p=subprocess.run(a,text=True,capture_output=True); print(p.stdout,end=''); print(p.stderr,end='');
 if check and p.returncode: raise RuntimeError('command failed: '+' '.join(a))
 return p
print('=== GPU PREFLIGHT ===')
s=sh(['nvidia-smi','--query-gpu=name,compute_cap,memory.total','--format=csv,noheader'],False)
first=s.stdout.splitlines()[0] if s.stdout.splitlines() else ''
if not first: raise RuntimeError('No NVIDIA GPU attached')
print(first)
is_p100='P100' in first.upper()
def probe():
 try:
  import torch; print(torch.__version__,torch.version.cuda,torch.cuda.get_device_name(0),torch.cuda.get_device_capability(0)); x=torch.ones((64,64),device='cuda',dtype=torch.float16); y=x@x; torch.cuda.synchronize(); print('CUDA kernel probe: PASS',float(y[0,0])); return True
 except Exception as e: print('CUDA kernel probe: FAIL',repr(e)); return False
ok=probe(); marker='/kaggle/working/.lily_torch_fixed'
if not ok and is_p100:
 print('Installing P100-compatible Torch 2.5.1/cu118...')
 sh([sys.executable,'-m','pip','install','--no-cache-dir','--force-reinstall','torch==2.5.1','torchvision==0.20.1','torchaudio==2.5.1','--index-url','https://download.pytorch.org/whl/cu118'])
 open(marker,'w').write('ok'); print('Restarting kernel. RUN CELL 1 AGAIN after reconnect.'); time.sleep(2); os._exit(0)
if not ok: raise RuntimeError('CUDA kernel stack still unusable')
print('READY FOR CELL 2')


In [ ]:
import os,sys,subprocess,time,shutil,json,secrets
from pathlib import Path
ROOT=Path('/kaggle/working/lily_wan_fast'); C=ROOT/'ComfyUI'; OUT=ROOT/'output'; OUT.mkdir(parents=True,exist_ok=True)
def run(a,cwd=None): print('>', ' '.join(map(str,a)),flush=True); subprocess.run(list(map(str,a)),cwd=cwd,check=True)
import torch; assert torch.cuda.is_available(); x=torch.ones(1,device='cuda'); torch.cuda.synchronize(); del x
print('[1/5] Installing runtime')
run([sys.executable,'-m','pip','install','-q','--upgrade','gradio==5.50.0','huggingface_hub==0.36.0','safetensors==0.6.2','sentencepiece==0.2.1','einops==0.8.1','torchsde==0.2.6','av==15.1.0'])
if not C.exists(): run(['git','clone','--depth','1','--branch','v0.3.59','https://github.com/Comfy-Org/ComfyUI.git',str(C)])
req=(C/'requirements.txt').read_text().splitlines(); safe=[]
for line in req:
 q=line.strip(); low=q.lower()
 if q and not q.startswith('#') and not low.startswith(('torch','torchvision','torchaudio')): safe.append(q)
if safe: run([sys.executable,'-m','pip','install','-q','--upgrade-strategy','only-if-needed',*safe])
print('[2/5] Downloading official Wan 2.2 5B files')
from huggingface_hub import hf_hub_download
repo='Comfy-Org/Wan_2.2_ComfyUI_Repackaged'; files=[('split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors',C/'models/diffusion_models'),('split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors',C/'models/text_encoders'),('split_files/vae/wan2.2_vae.safetensors',C/'models/vae')]
for remote,dest in files:
 dest.mkdir(parents=True,exist_ok=True); target=dest/Path(remote).name
 if not target.exists(): cached=hf_hub_download(repo_id=repo,filename=remote); os.symlink(cached,target)
 print('OK',target.name)
print('[3/5] Starting ComfyUI backend')
PORT=8188; env=os.environ.copy(); env['PYTORCH_CUDA_ALLOC_CONF']='max_split_size_mb:128'
cmd=[sys.executable,str(C/'main.py'),'--listen','127.0.0.1','--port',str(PORT),'--lowvram','--disable-xformers','--force-fp16','--fp32-vae','--reserve-vram','1.5']
log=open(ROOT/'comfy.log','w'); proc=subprocess.Popen(cmd,cwd=C,env=env,stdout=log,stderr=subprocess.STDOUT)
import requests
for _ in range(120):
 try:
  if requests.get(f'http://127.0.0.1:{PORT}/system_stats',timeout=2).ok: break
 except: pass
 if proc.poll() is not None: raise RuntimeError((ROOT/'comfy.log').read_text()[-5000:])
 time.sleep(1)
else: raise RuntimeError('ComfyUI startup timeout')
print('[4/5] Building mobile UI')
import gradio as gr
def upload_image(img):
 name=f'lily_{secrets.token_hex(6)}.png'; path=C/'input'/name; img.convert('RGB').save(path); return name
PRE={'⚡ TURBO':(320,192,33,8),'✨ NORMAL':(384,224,49,12),'👑 MAX':(448,256,65,18)}
def generate(img,prompt,preset,seed,progress=gr.Progress()):
 if img is None or not prompt.strip(): raise gr.Error('Upload an image and enter a prompt.')
 w,h,frames,steps=PRE[preset]; name=upload_image(img); sid=int(seed) if seed is not None else secrets.randbelow(2**31)
 wf={'1':{'class_type':'LoadImage','inputs':{'image':name}},'2':{'class_type':'UNETLoader','inputs':{'unet_name':'wan2.2_ti2v_5B_fp16.safetensors','weight_dtype':'default'}},'3':{'class_type':'CLIPLoader','inputs':{'clip_name':'umt5_xxl_fp8_e4m3fn_scaled.safetensors','type':'wan','device':'default'}},'4':{'class_type':'VAELoader','inputs':{'vae_name':'wan2.2_vae.safetensors'}},'5':{'class_type':'CLIPTextEncode','inputs':{'text':prompt,'clip':['3',0]}},'6':{'class_type':'CLIPTextEncode','inputs':{'text':'blurry, frozen, distorted, watermark, text, low quality','clip':['3',0]}},'7':{'class_type':'Wan22ImageToVideoLatent','inputs':{'positive':['5',0],'negative':['6',0],'vae':['4',0],'width':w,'height':h,'length':frames,'batch_size':1,'start_image':['1',0]}},'8':{'class_type':'ModelSamplingSD3','inputs':{'model':['2',0],'shift':5.0}},'9':{'class_type':'KSampler','inputs':{'model':['8',0],'seed':sid,'steps':steps,'cfg':5.0,'sampler_name':'uni_pc','scheduler':'simple','positive':['7',0],'negative':['7',1],'latent_image':['7',0],'denoise':1.0}},'10':{'class_type':'VAEDecodeTiled','inputs':{'samples':['9',0],'vae':['4',0],'tile_size':256,'overlap':64,'temporal_size':16,'temporal_overlap':4}},'11':{'class_type':'SaveAnimatedWEBP','inputs':{'images':['10',0],'filename_prefix':'lily_wan','fps':12.0,'lossless':False,'quality':90,'method':'default'}}
 r=requests.post(f'http://127.0.0.1:{PORT}/prompt',json={'prompt':wf},timeout=30)
 if not r.ok: raise gr.Error('Workflow rejected: '+r.text[:1600])
 pid=r.json()['prompt_id']; progress(0.05,desc='Wan is generating…')
 for i in range(3600):
  hist=requests.get(f'http://127.0.0.1:{PORT}/history/{pid}',timeout=10).json()
  if pid in hist:
   item=hist[pid]; imgs=item.get('outputs',{}).get('11',{}).get('images',[])
   if not imgs: raise gr.Error('Generation ended without output. '+json.dumps(item.get('status',{}))[:1200])
   f=imgs[0]; data=requests.get(f'http://127.0.0.1:{PORT}/view',params={'filename':f['filename'],'subfolder':f.get('subfolder',''),'type':f.get('type','output')},timeout=60).content
   webp=OUT/f'lily_{sid}.webp'; webp.write_bytes(data); mp4=OUT/f'lily_{sid}.mp4'
   run(['ffmpeg','-y','-loglevel','error','-i',str(webp),'-vf','fps=24,scale=640:-2:flags=lanczos','-c:v','libx264','-preset','veryfast','-crf','20','-pix_fmt','yuv420p','-movflags','+faststart',str(mp4)])
   progress(1,desc='Done'); return str(mp4),str(mp4),f'DONE · {preset} · {w}×{h} native → 640px MP4 · {frames} frames · {steps} steps · seed {sid}'
  time.sleep(2); progress(min(.9,.05+i/500),desc='Wan is generating…')
 raise gr.Error('Generation timed out')
css='.gradio-container{max-width:680px!important;margin:auto!important} button{min-height:52px}'
with gr.Blocks(css=css,title='Lily Wan 2.2 Fast Studio') as demo:
 gr.Markdown('# Lily Wan 2.2 Fast Studio\n**P100 fast path · Wan 2.2 TI2V 5B**')
 image=gr.Image(type='pil',label='Image'); prompt=gr.Textbox(lines=3,label='Motion prompt'); preset=gr.Radio(list(PRE),value='⚡ TURBO',label='Speed / quality'); seed=gr.Number(value=42,precision=0,label='Seed'); btn=gr.Button('GENERATE',variant='primary'); video=gr.Video(label='Video'); download=gr.File(label='MP4'); status=gr.Textbox(label='Status',interactive=False); btn.click(generate,[image,prompt,preset,seed],[video,download,status],concurrency_limit=1)
demo.queue(default_concurrency_limit=1,max_size=2)
print('[5/5] READY — opening share link'); demo.launch(share=True,server_name='0.0.0.0',prevent_thread_lock=True,allowed_paths=[str(OUT)])
